In [1]:
# Install required libraries quietly (-q hides dense logs) for models, data handling, tracking, and metrics

!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd  # Import pandas for tabular dataset structures (DataFrames)
import numpy as np  # Import numpy for quick array manipulations and math operations
from datasets import Dataset  # Import Hugging Face's Dataset format for high-speed training optimization

# Load the cleaned dataset from the local environment path using the Python fallback parser
df = pd.read_csv(
    'cleaned_train-3.csv',
    engine='python',
    on_bad_lines='warn'  # Print a warning notice if a row contains malformed content instead of crashing
)

# Define an explicit array holding the exact column targets for your multi-label toxicity categories
label_cols = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

# Convert the 6 numerical classification columns into a single floating-point tensor array list per text row
df['labels'] = df[label_cols].values.astype(float).tolist()

# Strip away raw metadata columns to keep only the processed text string and target label lists
df = df[['clean_text', 'labels']]

# Re-label the cleaned text key to match standardized Hugging Face conventions for transformer models
df = df.rename(columns={'clean_text': 'comment_text'})

# Convert the pandas structured DataFrame memory tree into an optimized Hugging Face Dataset format
dataset = Dataset.from_pandas(df)

# Split data into a Train partition (80%) and Evaluation partition (20%) using a fixed random seed for reproduceability
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

# Print out structural details (row counts, column layout properties) of the new split structure
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'labels'],
        num_rows: 127651
    })
    test: Dataset({
        features: ['comment_text', 'labels'],
        num_rows: 31913
    })
})


In [ ]:
# Check and display the overall grid boundary size (total row rows, total column properties) of your DataFrame
print(df.shape)

(159564, 2)


In [ ]:
from transformers import AutoTokenizer  # Import structural rules constructor to process raw text matrices

# Specify the architecture flavor of DistilBERT to fetch from the Hugging Face Hub registry
MODEL_NAME = "distilbert-base-uncased"

# Pull down vocabulary dictionaries and lookup configurations tied to the uncased base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Construct a helper mapping module to truncate long strings and pad short ones into fixed arrays
def tokenize(batch):
    return tokenizer(
        batch["comment_text"],  # Target the standardized text string dictionary key
        truncation=True,        # Chop down sentences that extend beyond the attention boundary window
        padding="max_length",   # Apply padding placeholders up to our specified limits
        max_length=128          # Fix sequence lengths uniformly to 128 tokens for standard GPU memory optimization
    )

# Distribute the tokenizing mapping workflow rapidly using multi-batch grouping over both dataset splits
tokenized = dataset.map(
    tokenize,
    batched=True
)

# Discard the original raw string column since it has been converted into numerical arrays
tokenized = tokenized.remove_columns(["comment_text"])

# Alter internal structural layouts to emit native PyTorch tensors for model processing
tokenized.set_format(
    type="torch",
    columns=[
        "input_ids",       # The specific vocabulary numbers mapping to each split word
        "attention_mask",  # Binary vector mapping out what is text data (1) versus pad filler tokens (0)
        "labels"           # Target answers for optimization calculations
    ]
)

# Output summary signatures verifying target dimensions and structural datatypes
print(tokenized)

Map:   0%|          | 0/127651 [00:00<?, ? examples/s]

Map:   0%|          | 0/31913 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 127651
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 31913
    })
})


In [ ]:
from transformers import AutoModelForSequenceClassification  # Import the standard model configuration head

# Fetch pre-trained model weights from the cloud and attach a custom output classification layer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,                                # Force output configuration to map exactly to the 6 target categories
    problem_type="multi_label_classification"   # Configure the network to use independent binary targets per class
)

# Send a visual notification message showing the initialization executed successfully
print("Model loaded")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [ ]:
import evaluate  # Import standard Hugging Face assessment evaluation modules
import numpy as np  # Bring in array formatting systems for evaluation computations
from sklearn.metrics import f1_score  # Import traditional calculation functions for class assessments
from scipy.special import expit  # Import the logistical sigmoid calculator to map outputs to probabilities

# Construct a function to convert log-odds outputs into classification scores during tracking steps
def compute_metrics(eval_pred):

    # Separate raw neural outputs (logits) from true target categories
    logits, labels = eval_pred

    # Map unbounded logit outputs into a standard probability window from 0.0 to 1.0 using a Sigmoid function
    probs = expit(logits)

    # Convert continuous probabilities into distinct binary choices (1 if probability > 50%, else 0)
    preds = (probs > 0.5).astype(int)

    # Compute and deliver performance metrics across all targets using an unweighted Macro Average
    return {
        "f1": f1_score(
            labels,
            preds,
            average="macro",      # Give equal importance weights to both rare and frequent toxicity categories
            zero_division=0       # Prevent mathematical crashes by returning 0 if a target receives no predictions
        )
    }

In [ ]:
from transformers import TrainingArguments  # Import hyperparameter tracking parameters configuration objects

# Establish core execution values for model optimization loops
training_args = TrainingArguments(
    output_dir="./results",           # Define the file tree folder tracking your historical checkpoint runs
    eval_strategy="epoch",            # Trigger validation score testing at the completion of every epoch pass
    save_strategy="epoch",            # Archive a model checkpoint folder to storage at the completion of every epoch pass
    learning_rate=2e-5,               # Establish the optimization step size (AdamW weight rate parameter)
    per_device_train_batch_size=16,   # Define the sample batch footprint processed simultaneously by training loops
    per_device_eval_batch_size=16,    # Define the sample batch footprint processed simultaneously by tracking loops
    num_train_epochs=5,               # Force training loops to execute 5 complete passes over the dataset
    weight_decay=0.01,                # Introduce regularization to penalize large weights and prevent overfitting
    load_best_model_at_end=True,      # Automatically roll back to your top performing checkpoint when training completes
    metric_for_best_model="f1",       # Use Macro F1-score performance as the primary baseline for evaluation grading
    greater_is_better=True,           # Tell the system that a higher evaluation target metric reflects superior results
    logging_steps=500,                # Output console debugging metrics frequently during training loops
    report_to="none"                  # Disable background telemetry reporting to external tracking platforms
)

In [ ]:
from transformers import Trainer  # Import the unified execution supervisor tool

# Package model elements, hyperparameter blocks, dataset sources, and evaluation code scripts into a Trainer object
trainer = Trainer(
    model=model,                      # Inject the target classification architecture
    args=training_args,               # Pass down the hyperparameter configurations
    train_dataset=tokenized["train"], # Assign the training data split
    eval_dataset=tokenized["test"],  # Assign the validation testing data split
    compute_metrics=compute_metrics   # Pass down the custom validation metric tracking script
)

In [ ]:
# Execute the fine-tuning operations across the active GPU acceleration hardware
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.039801,0.040006,0.606358
2,0.033176,0.037409,0.653468
3,0.022613,0.045476,0.683396
4,0.020023,0.049387,0.670228
5,0.012677,0.056208,0.668119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=39895, training_loss=0.02758115685244583, metrics={'train_runtime': 7829.8771, 'train_samples_per_second': 81.515, 'train_steps_per_second': 5.095, 'total_flos': 2.113850267548416e+16, 'train_loss': 0.02758115685244583, 'epoch': 5.0})

In [ ]:
# List local repository folders inside the results directory to verify step counts and checkpoint footprints
!ls results

checkpoint-15958  checkpoint-31916  checkpoint-7979
checkpoint-23937  checkpoint-39895


In [ ]:
# Export the model weight files into a clean folder for deployment execution
model.save_pretrained("/content/civicguard_distilbert")

# Export tokenizer matching indexes and mapping rules alongside the model weights
tokenizer.save_pretrained("/content/civicguard_distilbert")

# Confirm execution completion to the console screen interface
print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved


In [ ]:
import shutil  # Import standard system compression and file utility toolkits

# Package all production deployment configurations into a single, portable zip archive file
shutil.make_archive(
    "civicguard_distilbert",          # Name the target compressed output file
    "zip",                            # Set the compression algorithm target choice
    "/content/civicguard_distilbert"  # Identify the source folder path containing your model files
)

# Output success verification message
print("ZIP created")

ZIP created


In [ ]:
# Display the top validation metric score recorded across the training historical runs
print(trainer.state.best_metric)

# Output the precise file tree directory corresponding to that top performing validation checkpoint
print(trainer.state.best_model_checkpoint)

0.6833961823563185
./results/checkpoint-23937
